In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# --- CONFIGURATION ---
datasets = ['FD001', 'FD002', 'FD003', 'FD004']
results_svr = []

print("--- START METHOD 4: SUPPORT VECTOR REGRESSION (SVR) ---")
print("Note: SVR can be slow. We will use a sample of the training data to speed it up.")

for file_id in datasets:
    # 1. Load Data
    try:
        train_df = pd.read_csv(f'../data/processed/train_{file_id}_processed.csv')
        test_df = pd.read_csv(f'../data/processed/test_{file_id}_processed.csv')
        rul_df = pd.read_csv(f'../data/raw/RUL_{file_id}.txt', header=None, names=['RUL_actual'])
    except FileNotFoundError:
        print(f"Data not found for {file_id}. Skipping.")
        continue

    # 2. Engineering (Clipping)
    train_df['RUL'] = train_df['RUL'].clip(upper=125)
    y_true = rul_df['RUL_actual'].clip(upper=125)

    # 3. Features
    features = [col for col in train_df.columns if col not in ['unit_nr', 'time_cycles', 'RUL']]
    
    # 4. Normalization (CRUCIAL for SVR)
    # SVR fails if the data is not on the same scale
    scaler = StandardScaler()
    X_train_full = train_df[features]
    X_test = test_df[features]
    
    # Fit only on training data
    X_train_scaled = scaler.fit_transform(X_train_full)
    X_test_scaled = scaler.transform(X_test)
    y_train = train_df['RUL']

    # 5. TIME OPTIMIZATION (Sampling)
    # Datasets like FD002 have ~50,000 rows. SVR takes a long time (O(n^3)).
    # We will use 20% of the training data randomly to train the SVR.
    # This maintains representativeness but makes training viable in minutes.
    from sklearn.utils import resample
    X_train_sample, y_train_sample = resample(X_train_scaled, y_train, n_samples=10000, random_state=42, replace=False)
    # If the dataset is small (FD001), we use everything or the maximum available
    if len(X_train_scaled) < 10000:
        X_train_sample = X_train_scaled
        y_train_sample = y_train

    # 6. Train SVR
    # Kernel 'rbf' allows capturing non-linearities
    model = SVR(kernel='rbf', C=10, epsilon=5)
    
    print(f"-> Training SVR for {file_id} (Sample: {len(X_train_sample)} rows)...")
    model.fit(X_train_sample, y_train_sample)

    # 7. Evaluate
    test_last_cycle = test_df.groupby('unit_nr').last().reset_index()
    # Attention: we must normalize the test set too!
    X_test_last = test_last_cycle[features]
    X_test_last_scaled = scaler.transform(X_test_last)
    
    y_pred = model.predict(X_test_last_scaled)

    # Metrics
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    results_svr.append({
        'Dataset': file_id,
        'RMSE': rmse,
        'R2': r2
    })
    
    print(f"   -> {file_id}: RMSE = {rmse:.2f} | R2 = {r2:.4f}")

# --- FINAL PRESENTATION ---
results_df = pd.DataFrame(results_svr)
print("\n--- FINAL SUMMARY METHOD 4 (SVR) ---")
try:
    display(results_df)
except:
    print(results_df)

--- START METHOD 4: SUPPORT VECTOR REGRESSION (SVR) ---
Note: SVR can be slow. We will use a sample of the training data to speed it up.
-> Training SVR for FD001 (Sample: 10000 rows)...
   -> FD001: RMSE = 18.56 | R2 = 0.7854
-> Training SVR for FD002 (Sample: 10000 rows)...
   -> FD002: RMSE = 30.01 | R2 = 0.5116
-> Training SVR for FD003 (Sample: 10000 rows)...
   -> FD003: RMSE = 20.46 | R2 = 0.7271
-> Training SVR for FD004 (Sample: 10000 rows)...
   -> FD004: RMSE = 35.75 | R2 = 0.3082

--- FINAL SUMMARY METHOD 4 (SVR) ---


,Dataset,RMSE,R2
0,FD001,18.563205,0.785417
1,FD002,30.009832,0.511553
2,FD003,20.461761,0.727061
3,FD004,35.749152,0.308208
